In [1]:
import pandas as pd
import numpy as np
import math

file_path = "Customer_Final_Scores_FIXED.xlsx"
df = pd.read_excel(file_path)

print(df.head())
print(df.columns)

   Behavior_Spending_Score  Income_INR  Total_Credit_Limit  \
0                       73     2309818             1996830   
1                       26     2431785             2664150   
2                       66      488998             1618560   
3                       73     2043400              515757   
4                       73     1279111             2655214   

   Loan_Tenure_Months  Months_Since_Most_Recent_Delinquency  \
0                  48                                    56   
1                  24                                     6   
2                  72                                    21   
3                  24                                    54   
4                  24                                    32   

   Credit_Utilization_Ratio  Credit_Card_Utilization  Employment_Years  \
0                      0.15                     0.49                 0   
1                      0.53                     0.41                 8   
2                      0.2

In [2]:
# Default_y = 0 -> Non-Default (Good customer)
# Default_y = 1 -> Default (Bad customer)
goods = (df["Default_y"] == 0).sum()
bads = (df["Default_y"] == 1).sum()

print(f"Goods: {goods}, Bads: {bads}")

Goods: 70, Bads: 30


In [3]:
# STEP 3: Calculate Base Odds
BaseOdds = goods / bads
print(f"Base Odds = {BaseOdds:.2f}:1")

Base Odds = 2.33:1


In [4]:
# STEP 4: Choose PDO and Base Score
PDO = 20
BaseScore = 600


In [5]:
# STEP 5: Calculate Factor and Offset
Factor = PDO / math.log(2)
Offset = BaseScore - Factor * math.log(BaseOdds)

print(f"Factor = {Factor:.2f}, Offset = {Offset:.2f}")

Factor = 28.85, Offset = 575.55


In [6]:
# STEP 6: Create deciles on Final_Score
df["score_decile"] = pd.qcut(df["Final_Score"], 10, labels=False, duplicates="drop")

print(df["score_decile"].value_counts().sort_index())
print(df[["Final_Score", "score_decile", "Sample"]].head(10))

score_decile
0    10
1    10
2    11
3     9
4    10
5    10
6    10
7    10
8    10
9    10
Name: count, dtype: int64
   Final_Score  score_decile Sample
0          218             0  Train
1          836             7  Train
2          487             2   Test
3          722             4   Test
4          888             8  Train
5          508             2  Train
6          255             0  Train
7          900             8  Train
8          778             6  Train
9          704             4  Train


In [7]:
# STEP 7: Calculate stats per decile (full population, as intended for calibration)
decile_stats = df.groupby("score_decile").agg(
    total=("Final_Score", "size"),
    bads=("Default_y", "sum"),
    goods=("Default_y", lambda x: (x == 0).sum()),
    avg_score=("Final_Score", "mean")
).reset_index()

print(decile_stats)


   score_decile  total  bads  goods   avg_score
0             0     10     8      2  219.300000
1             1     10     8      2  410.900000
2             2     11     7      4  524.636364
3             3      9     2      7  617.666667
4             4     10     1      9  694.700000
5             5     10     2      8  749.900000
6             6     10     1      9  801.900000
7             7     10     0     10  843.500000
8             8     10     0     10  880.300000
9             9     10     1      9  960.900000


In [8]:
# STEP 8: Bad rate and odds per decile
decile_stats["p_bad"] = decile_stats["bads"] / decile_stats["total"]
decile_stats["odds"] = (1 - decile_stats["p_bad"]) / decile_stats["p_bad"]

print(decile_stats[["score_decile", "bads", "total", "p_bad", "odds"]])


   score_decile  bads  total     p_bad      odds
0             0     8     10  0.800000  0.250000
1             1     8     10  0.800000  0.250000
2             2     7     11  0.636364  0.571429
3             3     2      9  0.222222  3.500000
4             4     1     10  0.100000  9.000000
5             5     2     10  0.200000  4.000000
6             6     1     10  0.100000  9.000000
7             7     0     10  0.000000       inf
8             8     0     10  0.000000       inf
9             9     1     10  0.100000  9.000000


In [10]:
# STEP 9: Save results
output_file = "Customer_Final_Scores_with_Calibration_FIXED.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="RawData", index=False)
    decile_stats.to_excel(writer, sheet_name="DecileCalibration", index=False)

print(f"Calibration completed. Results saved to {output_file}")


Calibration completed. Results saved to Customer_Final_Scores_with_Calibration_FIXED.xlsx
